In [0]:
import requests
import pandas as pd
from pyspark.sql import SparkSession
from datetime import datetime

Fetch API

In [0]:
url="https://opensky-network.org/api/states/all"

response=requests.get(url)

data=response.json()

In [0]:
print(data)

Convert to DataFrame

In [0]:
columns=[
"icao24",
"callsign",
"origin_country",
"time_position",
"last_contact",
"longitude",
"latitude",
"baro_altitude",
"on_ground",
"velocity",
"true_track",
"vertical_rate",
"sensors",
"geo_altitude",
"squawk",
"spi",
"position_source"
]

pdf=pd.DataFrame(data["states"],columns=columns)

In [0]:
print(pdf)

Add Ingestion Metadata

Important for batch tracking.

Add:

ingestion timestamp batch date source

In [0]:
from pyspark.sql.functions import *


raw_df=(

spark.createDataFrame(pdf)

.withColumn(
"ingestion_timestamp",
current_timestamp()
)

.withColumn(
"batch_date",
current_date()
)

.withColumn(
"source",
lit("OpenSky API")
)

)

Convert NULL column to String

In [0]:
from pyspark.sql.functions import col

spark_df = raw_df.withColumn(
    "sensors",
    col("sensors").cast("string")
)

In [0]:
%sql
SHOW STORAGE CREDENTIALS;

In [0]:
%sql
LIST 'abfss://batch@uralibootcamp.dfs.core.windows.net/'

Create Metadata Table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS opensky.bronze.file_tracking
(
file_name STRING,
processed_timestamp TIMESTAMP,
status STRING
)

Generate File Name

Save Raw Data 

 Write CSV to ADLS

In [0]:
from datetime import datetime

file_name = (
    "flight_snapshot_"
    + datetime.now().strftime("%Y%m%d")
    + ".csv"
)

raw_path = "abfss://batch@uralibootcamp.dfs.core.windows.net/raw/Flights/"

output_path = raw_path + file_name


spark_df.write \
.mode("append") \
.option("header","true") \
.csv(output_path)